In [1]:
import pandas as pd
import numpy as np

from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import Trainer, TrainingArguments

import torch

d:\Placement Projects\NLP text summarizer\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

train_data = dataset["train"]
test_data = dataset["test"]
validation_data = dataset["validation"]

print(train_data)

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 287113
})


In [ ]:
# reduce dataset size
# train_data = train_data.select(range(5000))
# validation_data = validation_data.select(range(1000))
# test_data = test_data.select(range(1000))

In [19]:
train_data = train_data.select(range(500))
validation_data = validation_data.select(range(100))

In [20]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [21]:
def preprocess_function(examples):

    inputs = ["summarize: " + doc for doc in examples["article"]]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        examples["highlights"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [22]:
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_val = validation_data.map(preprocess_function, batched=True)
# tokenized_test = test_data.map(preprocess_function, batched=True)

Map: 100%|██████████| 100/100 [00:00<00:00, 159.14 examples/s]


In [23]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 915.45it/s]


In [24]:
training_args = TrainingArguments(
    output_dir="../models",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="../logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [25]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_val
)

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,2.439207


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.64s/it]
d:\Placement Projects\NLP text summarizer\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=125, training_loss=5.27140771484375, metrics={'train_runtime': 1663.0137, 'train_samples_per_second': 0.301, 'train_steps_per_second': 0.075, 'total_flos': 67670900736000.0, 'train_loss': 5.27140771484375, 'epoch': 1.0})

In [27]:
model.save_pretrained("../models/t5_summarizer")
tokenizer.save_pretrained("../models/t5_summarizer")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.51s/it]


('../models/t5_summarizer\\tokenizer_config.json',
 '../models/t5_summarizer\\tokenizer.json')

In [28]:
model = T5ForConditionalGeneration.from_pretrained("../models/t5_summarizer")

tokenizer = T5Tokenizer.from_pretrained("../models/t5_summarizer")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 883.96it/s]


In [29]:
def summarize(text):

    input_text = "summarize: " + text

    inputs = tokenizer.encode(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    summary_ids = model.generate(

        inputs,

        max_length=120,

        min_length=30,

        num_beams=4,

        length_penalty=2.0,

        early_stopping=True
    )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

In [30]:
article = test_data[0]["article"]

print("ARTICLE:\n")
print(article)

print("\nSUMMARY:\n")

print(summarize(article))

ARTICLE:

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednesday's ce

In [32]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

reference = test_data[0]["highlights"]

prediction = summarize(test_data[0]["article"])

scores = scorer.score(reference, prediction)

print(scores)

{'rouge1': Score(precision=0.25925925925925924, recall=0.4117647058823529, fmeasure=0.3181818181818182), 'rouge2': Score(precision=0.09433962264150944, recall=0.15151515151515152, fmeasure=0.11627906976744186), 'rougeL': Score(precision=0.18518518518518517, recall=0.29411764705882354, fmeasure=0.22727272727272727)}
